In [ ]:
import os
from dataclasses import dataclass
from typing import Literal

import dspy
import ipdb
import pandas as pd
from dotenv import load_dotenv
from dspy import ChainOfThought, InputField, Module, OutputField, Signature
from dspy.evaluate.metrics import answer_exact_match
from dspy.teleprompt import (
    BootstrapFewShot,
    BootstrapFewShotWithRandomSearch,
    LabeledFewShot,
)
from langchain_tavily import TavilySearch
from loguru import logger
from sklearn.model_selection import train_test_split

_ = load_dotenv()

In [ ]:
llm = dspy.LM(
    "deepseek/deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base="https://api.deepseek.com",
)

dspy.configure(lm=llm)
dspy.settings.configure(track_usage=True)

# Asking the same question three different ways

In [ ]:
class QA(Signature):
    question: str = InputField(desc="The question posed by the user")
    answer: str = OutputField(desc="The answer to the question")


qa = ChainOfThought(QA)
cot_response = qa(question="Explain the parallax effect")

In [ ]:
cot_response.get_lm_usage()

{}

In [ ]:
llm.inspect_history()





[2025-09-07T12:32:57.860328]

System message:

Your input fields are:
1. `question` (str): The question posed by the user
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): The answer to the question
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
Explain the parallax effect

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
The user is asking for an explanation of the parallax effect. This is a concept in astronomy and physics, where the apparent shift in the position of an object wh

In [ ]:
cot_response

Prediction(
    reasoning='The user is asking for an explanation of the parallax effect. This is a concept in astronomy and physics, where the apparent shift in the position of an object when viewed from different lines of sight is used to measure distances to stars. I should provide a clear, concise definition, explain how it works, mention its applications (especially in astronomy for measuring stellar distances), and perhaps give a simple everyday example (like viewing a finger with one eye closed vs. the other) to make it relatable.',
    answer="The parallax effect is the apparent shift in the position of an object when viewed from two different vantage points. This occurs because the observer's perspective changes, making nearby objects appear to move against a more distant background.\n\nIn astronomy, stellar parallax is used to measure distances to stars. As Earth orbits the Sun, a nearby star will seem to shift slightly relative to more distant stars. By measuring this angular

In [ ]:
class QAModule(Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(QA)

    def forward(self, question: str):
        return self.predictor(question=question)


qa = QAModule()
module_response = qa(question="Explain the parallax effect")

In [ ]:
module_response

Prediction(
    answer="The parallax effect is an optical phenomenon where the apparent position or direction of an object changes when viewed from different lines of sight. In astronomy, it refers to the apparent shift in the position of a nearby star against the background of more distant stars as observed from different points in Earth's orbit around the Sun. This shift is used to measure stellar distances, with one parsec defined as the distance at which a star would have a parallax angle of one arcsecond. In everyday contexts, parallax can be observed when looking at objects through a window while moving your head, or in design and web development, where it creates an illusion of depth by making background elements move slower than foreground elements as the user scrolls."
)

In [ ]:
qa = dspy.Predict("question -> answer")
basic_response = qa(question="Explain the parallax effect")

In [ ]:
basic_response

Prediction(
    answer='The parallax effect is an optical phenomenon where the apparent position or direction of an object appears to change when viewed from different perspectives. In astronomy, it refers to the apparent shift in the position of a nearby star against the background of more distant stars as the Earth orbits the Sun. This shift is used to measure stellar distances, with one parsec defined as the distance at which a star would have a parallax angle of one arcsecond. In everyday contexts, parallax can be observed when looking at objects through a window while moving—the closer objects seem to move more relative to distant ones. It is also utilized in photography, computer graphics, and web design to create depth and immersion.'
)

In [ ]:
llm.inspect_history(n=3)





[2025-09-07T12:32:57.860328]

System message:

Your input fields are:
1. `question` (str): The question posed by the user
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): The answer to the question
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
Explain the parallax effect

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
The user is asking for an explanation of the parallax effect. This is a concept in astronomy and physics, where the apparent shift in the position of an object wh

# Content Summarization

In [ ]:
document = """
# Tech Giant Announces Revolutionary Quantum Computing Breakthrough

**September 6, 2025 - Silicon Valley, CA**

In a groundbreaking announcement that could reshape the future of computing, TechNova Corporation revealed today that its research team has successfully developed a stable 1000-qubit quantum processor, marking a significant leap forward in quantum computing capabilities.

## The Breakthrough

The new quantum processor, dubbed "QuantumCore X1," operates at near-absolute zero temperatures and maintains quantum coherence for up to 10 minutes - a dramatic improvement from previous systems that could only sustain coherence for seconds. This advancement addresses one of the most significant challenges in quantum computing: quantum decoherence, where quantum states collapse due to environmental interference.

Dr. Sarah Chen, TechNova's Chief Quantum Scientist, explained during a press conference: "What we've achieved here isn't just an incremental improvement. We've fundamentally solved the stability problem that has plagued quantum systems for decades. Our proprietary error correction algorithms combined with advanced superconducting materials have created a quantum processor that's not only powerful but practically usable."

## Technical Specifications

The QuantumCore X1 features:
- 1000 fully-connected qubits
- 99.9% gate fidelity
- 10-minute coherence time
- Operating temperature of 0.01 Kelvin
- Custom-designed dilution refrigeration system
- Advanced error correction protocols

The processor is housed in a room-sized cryogenic system that maintains the extreme cold necessary for quantum operations. Despite its size, TechNova claims the system is 50% more energy-efficient than competing quantum computers.

## Potential Applications

Industry experts believe this breakthrough could revolutionize several fields:

**Cryptography and Security:** The quantum processor could break current encryption methods within hours, potentially rendering existing cybersecurity infrastructure obsolete. However, it could also enable quantum-resistant encryption protocols.

**Drug Discovery:** Pharmaceutical companies could simulate molecular interactions at unprecedented scales, potentially reducing drug development timelines from decades to years.

**Financial Modeling:** Complex risk assessments and portfolio optimizations that currently take weeks could be completed in minutes.

**Climate Science:** Weather prediction models could become exponentially more accurate, potentially forecasting weather patterns months in advance.

**Artificial Intelligence:** Machine learning algorithms could be accelerated beyond current capabilities, enabling AI systems to process and learn from data at quantum speeds.

## Industry Response

The announcement has sent shockwaves through the technology industry. Quantum computing rivals IBM and Google have yet to comment officially, but industry insiders suggest both companies are scrambling to accelerate their own quantum research programs.

"This changes everything," said Dr. Michael Rodriguez, a quantum computing researcher at Stanford University who was not involved in the project. "If TechNova's claims are verified, we're looking at a quantum advantage that's not theoretical but immediately practical. This could trigger the next industrial revolution."

Stock markets responded dramatically to the news, with TechNova's shares surging 47% in after-hours trading. Technology stocks across the board saw significant gains, while cybersecurity companies experienced volatility as investors grappled with the implications for current encryption methods.

## Challenges and Concerns

Despite the excitement, experts caution about potential challenges. The extreme operating conditions required for the quantum processor mean widespread deployment could be years away. Additionally, the implications for current cybersecurity infrastructure have raised concerns among government agencies worldwide.

The National Institute of Standards and Technology (NIST) issued a statement calling for "immediate collaboration with TechNova to understand the implications for national security and to accelerate the deployment of quantum-resistant encryption standards."

## Looking Forward

TechNova announced plans to make the QuantumCore X1 available to select research institutions and enterprise partners starting in Q2 2026. The company is also working on a cloud-based quantum computing service that would allow organizations to access quantum processing power remotely without needing to install the complex hardware systems.

CEO Robert Kim concluded the announcement by saying: "Today marks the beginning of the quantum age. We're not just building faster computers; we're unlocking computational possibilities that seemed like science fiction just years ago. The problems we'll be able to solve and the discoveries we'll make will benefit all of humanity."

The scientific community awaits peer review of TechNova's claims, with several independent verification studies already being planned. If confirmed, this breakthrough could indeed mark a pivotal moment in the history of computing technology.

---

*This story is developing. More details will be added as they become available.*

**Contact:** tech.reporter@newsnetwork.com | Follow us @TechNewsDaily
"""

In [ ]:
class DocumentSummarizer(Signature):
    document: str = InputField(desc="The news article to be summarised")
    summary: str = OutputField(desc="The summary of the document")


doc_summarizer = ChainOfThought(DocumentSummarizer)

document_summary = doc_summarizer(document=document)
document_summary

Prediction(
    reasoning='The document is a news article announcing a major quantum computing breakthrough by TechNova Corporation. To summarize it effectively, I need to capture the key points: the announcement of a stable 1000-qubit quantum processor (QuantumCore X1), its technical specifications (e.g., 10-minute coherence time), the significance (solving decoherence), potential applications (cryptography, drug discovery, etc.), industry and market reactions, challenges (e.g., deployment and security concerns), and future plans (availability in 2026). The summary should be concise, focusing on the main event and its implications without extraneous details.',
    summary='TechNova Corporation has announced the development of the QuantumCore X1, a stable 1000-qubit quantum processor that maintains quantum coherence for 10 minutes, a major advancement addressing decoherence. This breakthrough could revolutionize fields like cryptography, drug discovery, and AI, though challenges in dep

In [ ]:
document_summary.summary

'TechNova Corporation has announced the development of the QuantumCore X1, a stable 1000-qubit quantum processor that maintains quantum coherence for 10 minutes, a major advancement addressing decoherence. This breakthrough could revolutionize fields like cryptography, drug discovery, and AI, though challenges in deployment and cybersecurity implications remain. The processor is set to be available to select partners in 2026, with significant industry and market reactions following the news.'

## Verify Contextual Alignment

In [ ]:
class ContextualAlignment(Signature):
    context: str = InputField(desc="Retrieved context passages or documents")
    text: str = InputField(desc="Generated or retrieved text to evaluate")
    relevance: Literal["relevant", "irrelevant"] = OutputField(
        desc="Does the text address the same topic as the context?"
    )
    groundedness: Literal["grounded", "hallucinated"] = OutputField(
        desc="Is the text supported by the context?"
    )
    completeness: Literal["complete", "partial"] = OutputField(
        desc="Does the text cover the main points of the context?"
    )
    completeness_reason: str = OutputField(
        desc="If completeness is 'partial', explain what was missing"
    )
    evidence: str = OutputField(
        desc="Cited phrases, passages, or references from the context that support the evaluation"
    )
    confidence: float = OutputField(
        desc="Confidence score reflecting certainty of the evaluation", gt=0.0, lt=1.0
    )

In [ ]:
context_text = """
The parallax effect is an apparent shift in the position of an object
when viewed from different perspectives. Astronomers use it to measure
distances to nearby stars.
"""

generated_text = "The parallax effect helps astronomers estimate star distances by observing apparent shifts."
alignment_checker = dspy.Predict(ContextualAlignment)
response = alignment_checker(context=context_text, text=generated_text)

In [ ]:
print(f"Relevance: {response.relevance}")
print(f"Groundedness: {response.groundedness}")
print(f"Completeness: {response.completeness}")
print(f"Completeness Reason: {response.completeness_reason}")
print(f"Evidence: {response.evidence}")
print(f"Confidence: {response.confidence}")

Relevance: relevant
Groundedness: grounded
Completeness: partial
Completeness Reason: The text mentions the use of parallax for estimating star distances and observing apparent shifts, but it omits the key detail that this is specifically for nearby stars and does not explicitly state that the shift is due to viewing from different perspectives.
Evidence: The context states: "The parallax effect is an apparent shift in the position of an object when viewed from different perspectives. Astronomers use it to measure distances to nearby stars." The text reflects "helps astronomers estimate star distances by observing apparent shifts," aligning with the context but missing the specificity of "nearby stars" and the cause "viewed from different perspectives."
Confidence: 0.95


In [ ]:
def search(query: str, k: int = 5):
    tavily = TavilySearch(max_results=k, topic="general")
    responses = tavily.invoke(query)
    results = responses.get("results", [])
    return "\n".join([result["content"] for result in results])


class Hop(dspy.Module):
    def __init__(self, num_docs=4, num_hops=2):
        self.num_docs, self.num_hops = num_docs, num_hops
        self.generate_query = dspy.ChainOfThought("claim, notes -> query")
        self.append_notes = dspy.ChainOfThought(
            "claim, notes, context -> new_notes: list[str], titles: list[str]"
        )

    def forward(self, claim: str) -> dspy.Prediction:
        notes = []
        titles = []

        for _ in range(self.num_hops):
            ipdb.set_trace()
            notes_str = "; ".join(notes) if notes else "No notes yet"
            query = self.generate_query(claim=claim, notes=notes_str).query
            context = search(query, k=self.num_docs)
            prediction = self.append_notes(
                claim=claim, notes=notes_str, context=context
            )

            new_notes = [note for note in prediction.new_notes if note not in notes]
            new_titles = [title for title in prediction.titles if title not in titles]

            notes.extend(new_notes)
            titles.extend(new_titles)

        return dspy.Prediction(notes=notes, titles=titles)

In [ ]:
hop = Hop()
answer = hop(
    claim="Stephen Curry is the best 3 pointer shooter ever in the human history"
)

> /tmp/ipykernel_87459/246189426.py(22)forward()
     21             ipdb.set_trace()
---> 22             notes_str = "; ".join(notes) if notes else "No notes yet"
     23             query = self.generate_query(claim=claim, notes=notes_str).query



> /tmp/ipykernel_87459/246189426.py(22)forward()
     21             ipdb.set_trace()
---> 22             notes_str = "; ".join(notes) if notes else "No notes yet"
     23             query = self.generate_query(claim=claim, notes=notes_str).query



In [ ]:
answer

Prediction(
    notes=['Stephen Curry is the all-time leader in career 3-pointers made with 3117, surpassing Ray Allen (2973).', 'Curry has a career 3-point percentage of 43.3%, higher than Ray Allen (40.0%) and Reggie Miller (39.5%).', "Curry averages 8.3 three-point attempts per game, compared to Allen's 5.7, indicating high volume with elite efficiency.", "Curry's playing style and success have changed the way basketball is played and influenced young players globally."],
    titles=['NBA All-Time 3-Point Leader', '3-Point Shooting Efficiency', 'Impact on Basketball Evolution', 'Stephen Curry: The Greatest 3-Point Shooter in History', 'Unmatched 3-Point Records and Efficiency', "Impact and Legacy of Curry's Shooting Prowess"]
)

In [ ]:
llm.inspect_history()





[2025-09-07T12:47:53.574518]

System message:

Your input fields are:
1. `claim` (str): 
2. `notes` (str): 
3. `context` (str):
Your output fields are:
1. `reasoning` (str): 
2. `new_notes` (list[str]): 
3. `titles` (list[str]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## claim ## ]]
{claim}

[[ ## notes ## ]]
{notes}

[[ ## context ## ]]
{context}

[[ ## reasoning ## ]]
{reasoning}

[[ ## new_notes ## ]]
{new_notes}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## titles ## ]]
{titles}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `claim`, `notes`, `context`, produce the fields `new_notes`, `titles`.


User message:

[[ ## claim ## ]]
Stephen Curry is the best 3 pointer shooter ever 

# Evaluation

In [ ]:
qa_pair = dspy.Example(question="What is your name?", answer="My name is Chris")
print(qa_pair)
print(qa_pair.question)
print(qa_pair.answer)

Example({'question': 'What is your name?', 'answer': 'My name is Chris'}) (input_keys=None)
What is your name?
My name is Chris


In [ ]:
print(qa_pair.with_inputs("question"))

Example({'question': 'What is your name?', 'answer': 'My name is Chris'}) (input_keys={'question'})


In [ ]:
article_summary = dspy.Example(
    article="This is an article.", summary="This is a summary."
).with_inputs("article")
input_key_only = article_summary.inputs()
output_key_only = article_summary.labels()

print(f"input key: {input_key_only}")
print(f"Output Key: {output_key_only}")

input key: Example({'article': 'This is an article.'}) (input_keys={'article'})
Output Key: Example({'summary': 'This is a summary.'}) (input_keys=None)


# Metrics

In [ ]:
def validate_answer(example, pred, trace=None):
    answer_match = example.answer.lower() == pred.answer.lower()

    context_match = any((pred.answer.lower() in c) for c in pred.context)
    if trace is None:
        return (answer_match + context_match) / 2.0
    else:
        return answer_match and context_match

In [ ]:
# Define the signature for automatic assessments.


class Assess(dspy.Signature):
    """Assess the quality of a tweet along the specified dimension."""

    assessed_text = dspy.InputField()
    assessment_question = dspy.InputField()
    assessment_answer: bool = dspy.OutputField()

In [ ]:
@dataclass
class TestSample:
    question: str
    answer: str
    output: str

In [ ]:
def metric(gold, pred, trace=None):
    question, answer, tweet = gold.question, gold.answer, pred.output

    engaging = "Does the assessed text make for a self-contained, engaging tweet?"
    correct = f"The text should answer `{question}` with `{answer}`. Does the assessed text contain this answer?"

    correct = dspy.Predict(Assess)(assessed_text=tweet, assessment_question=correct)
    engaging = dspy.Predict(Assess)(assessed_text=tweet, assessment_question=engaging)

    correct, engaging = (m.assessment_answer for m in [correct, engaging])
    score = (correct + engaging) if correct and (len(tweet) <= 280) else 0

    if trace is not None:
        return score >= 2
    return score / 2.0

In [ ]:
logger.info("Testing DSPy ")

gold = TestSample(
    question="What algorithm powers GPT models?",
    answer="Transformer",
    output="Transformer is the backbone of GPT models. Attention is all you need #MachineLearning #AI",
)

pred = TestSample(
    question="What algorithm powers GPT models?",
    answer="Transformer",
    output="Transformer is the backbone of GPT models. Attention is all you need #MachineLearning #AI",
)

try:
    score = metric(gold, pred)
    logger.info(f"Assessment score: {score}")
except Exception as e:
    logger.error(f"Assessment failed: {e}")

2025-09-07 12:53:43.264 | INFO     | __main__:<module>:1 - Testing DSPy 🤖
2025-09-07 12:53:51.877 | INFO     | __main__:<module>:17 - Assessment score: 1.0


# Optimization

In [ ]:
df = pd.read_csv("data/sliced_tweets.csv")

In [ ]:
df["target"] = df["target"].apply(lambda x: "Positive" if x == 4 else "Negative")
df = df.rename({"target": "polarity"}, axis=1)
df

,tweet,polarity
0,"I'm getting back into rythm now, but there may...",Negative
1,I think I deserve Chipotle on a lazy day like ...,Positive
2,painted his garage door with his lovely wife o...,Positive
3,Just been sunbaking. I feel nice and brown LOL...,Negative
4,@trooper346 I forgot it was a girl Carol R. C...,Positive
...,...,...
95,opening wine - Charlton have been relegated,Negative
96,I Joined ShoeDazzle Society by @KimKardashian ...,Positive
97,@TissieTC I agree with you!.. @nick_carter doe...,Positive
98,Wants Paul To Get Better Soon &lt;3,Positive


In [ ]:
trainset, testset = train_test_split(df, test_size=0.8, random_state=42)

testset

,tweet,polarity
83,"@DannyGirlAlways I will ttyl, I feel bad for l...",Negative
53,"@remzology the audacity of hope, perhaps?",Positive
70,we lost in netball but we'll win in tennis to...,Negative
45,Follow me if any of you have ever adopted a pe...,Positive
44,Car broken,Negative
...,...,...
57,@thethomas Im fine how are you?,Positive
75,@KeithFollett Actually worse...,Negative
32,I just wrote the most useless essay I have eve...,Negative
94,Ok ok so I got on the bus buh I'll be sooo lat...,Negative


In [ ]:
train = [
    dspy.Example(tweet=x["tweet"], answer=x["polarity"]).with_inputs("tweet")
    for x in trainset.to_dict(orient="records")
]
test = [
    dspy.Example(tweet=x["tweet"], answer=x["polarity"]).with_inputs("tweet")
    for x in testset.to_dict(orient="records")
]

In [ ]:
class TweetPolarity(Signature):
    tweet: str = InputField(
        desc="The text of the tweet to analyze for sentiment polarity"
    )
    answer: Literal["Positive", "Negative"] = OutputField(
        desc="Predicted sentiment polarity of the tweet"
    )


prediction = dspy.Predict(TweetPolarity)

In [ ]:
evaluate_program = dspy.Evaluate(
    devset=test,
    metric=answer_exact_match,  # Expects an answer attribute
    num_threads=1,
    display_progress=True,
    display_table=10,
    provide_traceback=True,
)

In [ ]:
# Baseline

eval = evaluate_program(prediction)
eval

  0%|          | 0/80 [00:00<?, ?it/s]

Average Metric: 60.00 / 80 (75.0%): 100%|██████████| 80/80 [00:53<00:00,  1.49it/s]

2025/09/07 17:27:29 INFO dspy.evaluate.evaluate: Average Metric: 60 / 80 (75.0%)


,tweet,example_answer,pred_answer,answer_exact_match
0,"@DannyGirlAlways I will ttyl, I feel bad for leaving you",Negative,Negative,✔️ [True]
1,"@remzology the audacity of hope, perhaps?",Positive,Positive,✔️ [True]
2,we lost in netball but we'll win in tennis tomorrow.,Negative,Positive,
3,Follow me if any of you have ever adopted a pet from an animal she...,Positive,Positive,✔️ [True]
4,Car broken,Negative,Negative,✔️ [True]
5,@alaksir thank you pak,Positive,Positive,✔️ [True]
6,in the new kingdom hearts for DS will it be possible to play as Ve...,Positive,Positive,✔️ [True]
7,@saywhatx I love you moree&lt;3,Positive,Positive,✔️ [True]
8,@aplusk Hey! They did a section of MalariaNoMore on BBC Breakfast ...,Negative,Negative,✔️ [True]
9,"I'm getting back into rythm now, but there may not be enough time ...",Negative,Negative,✔️ [True]


EvaluationResult(score=75.0, results=<list of 80 results>)

# Labelled Few Shot

In [ ]:
lfs_optimizer = LabeledFewShot(k=16)  # Use 16 examples in prompts

lfs_compiled = lfs_optimizer.compile(prediction, trainset=train)

In [ ]:
lfs_eval = evaluate_program(
    lfs_compiled
)  # It actually performs better than the baseline with an accuracy of 76.2

Average Metric: 61.00 / 80 (76.2%): 100%|██████████| 80/80 [00:50<00:00,  1.60it/s]

2025/09/07 18:11:05 INFO dspy.evaluate.evaluate: Average Metric: 61 / 80 (76.2%)


,tweet,example_answer,pred_answer,answer_exact_match
0,"@DannyGirlAlways I will ttyl, I feel bad for leaving you",Negative,Negative,✔️ [True]
1,"@remzology the audacity of hope, perhaps?",Positive,Positive,✔️ [True]
2,we lost in netball but we'll win in tennis tomorrow.,Negative,Negative,✔️ [True]
3,Follow me if any of you have ever adopted a pet from an animal she...,Positive,Positive,✔️ [True]
4,Car broken,Negative,Negative,✔️ [True]
5,@alaksir thank you pak,Positive,Positive,✔️ [True]
6,in the new kingdom hearts for DS will it be possible to play as Ve...,Positive,Positive,✔️ [True]
7,@saywhatx I love you moree&lt;3,Positive,Positive,✔️ [True]
8,@aplusk Hey! They did a section of MalariaNoMore on BBC Breakfast ...,Negative,Negative,✔️ [True]
9,"I'm getting back into rythm now, but there may not be enough time ...",Negative,Negative,✔️ [True]


# Bootstrap FewShot Prompting

In [ ]:
teleprompter = BootstrapFewShot(
    metric=answer_exact_match,
    max_labeled_demos=16,
    max_bootstrapped_demos=4,
    metric_threshold=1,
)
bootstrap_compiled = teleprompter.compile(prediction, trainset=train)

 20%|██        | 4/20 [00:19<01:17,  4.87s/it]

Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.


In [ ]:
bootstrap_eval = evaluate_program(bootstrap_compiled)
bootstrap_eval  # Performs just as good as our baseline model

  0%|          | 0/80 [00:00<?, ?it/s]

Average Metric: 60.00 / 80 (75.0%): 100%|██████████| 80/80 [00:49<00:00,  1.62it/s]

2025/09/07 18:27:33 INFO dspy.evaluate.evaluate: Average Metric: 60 / 80 (75.0%)


,tweet,example_answer,pred_answer,answer_exact_match
0,"@DannyGirlAlways I will ttyl, I feel bad for leaving you",Negative,Negative,✔️ [True]
1,"@remzology the audacity of hope, perhaps?",Positive,Positive,✔️ [True]
2,we lost in netball but we'll win in tennis tomorrow.,Negative,Negative,✔️ [True]
3,Follow me if any of you have ever adopted a pet from an animal she...,Positive,Positive,✔️ [True]
4,Car broken,Negative,Negative,✔️ [True]
5,@alaksir thank you pak,Positive,Positive,✔️ [True]
6,in the new kingdom hearts for DS will it be possible to play as Ve...,Positive,Negative,
7,@saywhatx I love you moree&lt;3,Positive,Positive,✔️ [True]
8,@aplusk Hey! They did a section of MalariaNoMore on BBC Breakfast ...,Negative,Negative,✔️ [True]
9,"I'm getting back into rythm now, but there may not be enough time ...",Negative,Negative,✔️ [True]


EvaluationResult(score=75.0, results=<list of 80 results>)

# BootstrappedFewShotWithRandomSearch

In [ ]:
bsfswrs_optimizer = BootstrapFewShotWithRandomSearch(
    metric=answer_exact_match,
    num_candidate_programs=16,
    max_bootstrapped_demos=5,
    max_labeled_demos=16,
)

bsfswrs_compiled = bsfswrs_optimizer.compile(prediction, trainset=train)

Going to sample between 1 and 5 traces per predictor.
Will attempt to bootstrap 16 candidate sets.
Average Metric: 14.00 / 20 (70.0%): 100%|██████████| 20/20 [00:15<00:00,  1.25it/s]

2025/09/07 18:47:40 INFO dspy.evaluate.evaluate: Average Metric: 14 / 20 (70.0%)



New best score: 70.0 for seed -3
Scores so far: [70.0]
Best score so far: 70.0
Average Metric: 19.00 / 20 (95.0%): 100%|██████████| 20/20 [00:14<00:00,  1.39it/s] 

2025/09/07 18:47:55 INFO dspy.evaluate.evaluate: Average Metric: 19 / 20 (95.0%)



New best score: 95.0 for seed -2
Scores so far: [70.0, 95.0]
Best score so far: 95.0


...


Scores so far: [70.0, 95.0, 95.0, 95.0, 95.0, 95.0, 100.0, 100.0, 90.0, 95.0, 85.0, 95.0, 100.0, 85.0, 100.0, 90.0, 95.0, 100.0]
Best score so far: 100.0


 15%|█▌        | 3/20 [00:13<01:17,  4.56s/it]


Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Average Metric: 18.00 / 20 (90.0%): 100%|██████████| 20/20 [00:12<00:00,  1.55it/s]

2025/09/07 18:56:35 INFO dspy.evaluate.evaluate: Average Metric: 18 / 20 (90.0%)



Scores so far: [70.0, 95.0, 95.0, 95.0, 95.0, 95.0, 100.0, 100.0, 90.0, 95.0, 85.0, 95.0, 100.0, 85.0, 100.0, 90.0, 95.0, 100.0, 90.0]
Best score so far: 100.0
19 candidate programs found.


In [ ]:
bsfswrs_eval = evaluate_program(bsfswrs_compiled)
bsfswrs_eval  # Performs just as good as our baseline model

Average Metric: 60.00 / 80 (75.0%): 100%|██████████| 80/80 [00:47<00:00,  1.69it/s]

2025/09/07 18:58:44 INFO dspy.evaluate.evaluate: Average Metric: 60 / 80 (75.0%)


,tweet,example_answer,pred_answer,answer_exact_match
0,"@DannyGirlAlways I will ttyl, I feel bad for leaving you",Negative,Negative,✔️ [True]
1,"@remzology the audacity of hope, perhaps?",Positive,Positive,✔️ [True]
2,we lost in netball but we'll win in tennis tomorrow.,Negative,Negative,✔️ [True]
3,Follow me if any of you have ever adopted a pet from an animal she...,Positive,Positive,✔️ [True]
4,Car broken,Negative,Negative,✔️ [True]
5,@alaksir thank you pak,Positive,Positive,✔️ [True]
6,in the new kingdom hearts for DS will it be possible to play as Ve...,Positive,Negative,
7,@saywhatx I love you moree&lt;3,Positive,Positive,✔️ [True]
8,@aplusk Hey! They did a section of MalariaNoMore on BBC Breakfast ...,Negative,Negative,✔️ [True]
9,"I'm getting back into rythm now, but there may not be enough time ...",Negative,Negative,✔️ [True]


EvaluationResult(score=75.0, results=<list of 80 results>)